In [1]:
import os
import pandas as pd
import numpy as np
import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity


def locate_file(relative_path: str) -> str:
    """
    Search common Kaggle directories for the given relative file path.
    Returns the first existing full path or raises FileNotFoundError.
    """
    search_roots = [
        "./data",
        "./input",
        "/kaggle/input",
        "./kaggle/input",
        "data",
        "input",
    ]
    for root in search_roots:
        candidate = os.path.join(root, relative_path)
        if os.path.exists(candidate):
            return candidate
    raise FileNotFoundError(f"Could not find {relative_path} in any known location.")


train_path = locate_file("chaii-hindi-and-tamil-question-answering/train.csv")
test_path = locate_file("chaii-hindi-and-tamil-question-answering/test.csv")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)




In [2]:
# Combine question and context for the primary TF‑IDF space
train_df["qa_text"] = (
    train_df["question"].astype(str) + " " + train_df["context"].astype(str)
)
test_df["qa_text"] = (
    test_df["question"].astype(str) + " " + test_df["context"].astype(str)
)

# Primary TF‑IDF vectorizer (question + context)
vectorizer = TfidfVectorizer(
    lowercase=True,
    token_pattern=r"(?u)\b\w+\b",
    stop_words=None,
    ngram_range=(1, 2),
    sublinear_tf=True,
)

train_questions_tfidf = vectorizer.fit_transform(train_df["qa_text"])

# Secondary TF‑IDF vectorizer for questions only (used for cosine similarity)
vectorizer_q = TfidfVectorizer(
    lowercase=True,
    token_pattern=r"(?u)\b\w+\b",
    stop_words=None,
    ngram_range=(1, 2),
    sublinear_tf=True,
)

train_questions_only_tfidf = vectorizer_q.fit_transform(train_df["question"])

# Nearest‑Neighbour model on the primary space
nn_model = NearestNeighbors(n_neighbors=200, metric="cosine")
nn_model.fit(train_questions_tfidf)




NearestNeighbors(metric='cosine', n_neighbors=200)

In [3]:
def jaccard(str1: str, str2: str) -> float:
    a = set(str1.lower().split())
    b = set(str2.lower().split())
    c = a.intersection(b)
    return (
        float(len(c)) / (len(a) + len(b) - len(c))
        if (len(a) + len(b) - len(c)) > 0
        else 0.0
    )


def nearest_answer(qa_text: str, question: str, language: str, context: str) -> str:
    """
    Retrieve the most suitable answer from the training set.
    - Fetch up to 200 neighbours using the primary QA TF‑IDF vectors.
    - Keep the top 50 for efficiency.
    - Apply language filtering.
    - Re‑rank candidates using a blend of:
        * Jaccard similarity between the candidate answer and the test question.
        * Cosine similarity between the test question and the candidate's training question.
    - Return the answer with the highest combined score, falling back to the closest neighbour.
    """
    # Retrieve neighbours in the primary space
    q_vec = vectorizer.transform([qa_text])
    distances_qa, indices = nn_model.kneighbors(q_vec, return_distance=True)
    candidate_idxs = indices[0][:50]

    # Language filter
    lang_mask = train_df.iloc[candidate_idxs]["language"] == language
    if lang_mask.any():
        candidate_idxs = np.array(candidate_idxs)[lang_mask.values]

    # Pre‑compute cosine similarity of test question vs candidates' questions
    q_vec_question = vectorizer_q.transform([question])
    candidate_question_tfidf = train_questions_only_tfidf[candidate_idxs]
    cos_sims = cosine_similarity(q_vec_question, candidate_question_tfidf).flatten()

    best_idx = None
    best_score = -1.0

    for rank, idx in enumerate(candidate_idxs):
        ans = train_df.iloc[int(idx)]["answer_text"]
        if pd.isna(ans):
            continue
        # Jaccard between answer and test question
        jac = jaccard(str(ans), question)
        # Cosine similarity between questions
        cos = cos_sims[rank]
        # Weighted blend (equal weight)
        combined = 0.5 * jac + 0.5 * cos
        if combined > best_score:
            best_score = combined
            best_idx = idx

    if best_idx is not None:
        return str(train_df.iloc[int(best_idx)]["answer_text"]).strip()

    # Fallback: return answer of the absolute closest neighbour
    fallback_idx = indices[0][0]
    fallback_ans = train_df.iloc[int(fallback_idx)]["answer_text"]
    return "" if pd.isna(fallback_ans) else str(fallback_ans).strip()




In [4]:
predictions = []
for _, row in tqdm.tqdm(test_df.iterrows(), total=len(test_df)):
    pred = nearest_answer(
        row["qa_text"], row["question"], row["language"], row["context"]
    )
    predictions.append(pred.strip())
assert len(predictions) == len(test_df), "Prediction length mismatch"




100%|██████████| 112/112 [00:03<00:00, 31.58it/s]


In [5]:
submission = pd.DataFrame({"id": test_df["id"], "PredictionString": predictions})
submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission saved to {submission_path}, shape: {submission.shape}")

Submission saved to submission.csv, shape: (112, 2)
